In [11]:
import sys
import os
import importlib.util
import sqlite3
import pandas as pd
import sys
import pycountry


parent_dir = os.path.abspath('../')
scraper_path = os.path.join(parent_dir)
sys.path.append(scraper_path)

In [2]:
from scrappers.article_extractor import ArticleExtractor

In [3]:
from scrappers.rss_scraper import RssScraper

   
scraper = RssScraper()

# frequency: 1=hourly, 2=every 4 hours, 3=every 6 hours, 4=daily
scraper.add_source("CNN", "http://rss.cnn.com/rss/cnn_topstories.rss", 1)
scraper.add_source("BBC", "http://feeds.bbci.co.uk/news/rss.xml", 1)
scraper.add_source("NYT", "https://rss.nytimes.com/services/xml/rss/nyt/HomePage.xml", 2)

2025-04-12 10:22:04,497 - rss_scraper - WARNING - Source with URL http://rss.cnn.com/rss/cnn_topstories.rss already exists
2025-04-12 10:22:04,499 - rss_scraper - WARNING - Source with URL http://feeds.bbci.co.uk/news/rss.xml already exists
2025-04-12 10:22:04,500 - rss_scraper - WARNING - Source with URL https://rss.nytimes.com/services/xml/rss/nyt/HomePage.xml already exists


In [4]:
data = scraper.scrape_feeds(force=True)

2025-04-12 10:22:06,424 - rss_scraper - INFO - Found 4 sources
2025-04-12 10:22:06,425 - rss_scraper - INFO - Scraping feed: CNN (http://rss.cnn.com/rss/cnn_topstories.rss)
2025-04-12 10:22:07,265 - rss_scraper - INFO - Found 16 articles from CNN
2025-04-12 10:22:09,276 - rss_scraper - INFO - Scraping feed: BBC (http://feeds.bbci.co.uk/news/rss.xml)
2025-04-12 10:22:09,483 - rss_scraper - INFO - Found 35 articles from BBC
2025-04-12 10:22:11,495 - rss_scraper - INFO - Scraping feed: NYT (https://rss.nytimes.com/services/xml/rss/nyt/HomePage.xml)
2025-04-12 10:22:12,607 - rss_scraper - INFO - Found 23 articles from NYT
2025-04-12 10:22:14,619 - rss_scraper - INFO - Scraping feed: New Scientist (https://www.newscientist.com/feed/home/?cmpid=RSS%7CNSNS-Home)
2025-04-12 10:22:21,002 - rss_scraper - INFO - Found 51 articles from New Scientist
2025-04-12 10:22:23,018 - rss_scraper - INFO - Completed scraping, found 125 new articles
2025-04-12 10:22:23,040 - rss_scraper - INFO - Found 105 art

In [6]:
DB_FILE = os.path.join("../scrappers/articles.db")
conn = sqlite3.connect(DB_FILE)
cursor = conn.cursor()

df = pd.read_sql_query("SELECT * FROM articles WHERE status='COMPLETE'" , con=conn)


#extract complete content and looks for country names in each news article

creat a new column with all country names present in the content.

In [24]:
completed_df = df[df['status']=='COMPLETE']
country_names = [country.name for country in pycountry.countries]
query = '|'.join(country_names)

df['query_match'] = completed_df['content'].str.lower().str.contains(query)
df['query_match'] = completed_df['content'].str.contains(query, case=False)

/tmp/ipykernel_350983/2085291202.py:5: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  df['query_match'] = completed_df['content'].str.lower().str.contains(query)
/tmp/ipykernel_350983/2085291202.py:6: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  df['query_match'] = completed_df['content'].str.contains(query, case=False)


In [40]:
completed_df

,id,source_id,title,url,published_date,content,created_at,html_content,extraction_date,status,query_match
0,77,2,Treasury minister Darren Jones says globalisat...,https://www.bbc.com/news/articles/ckg10yjp7meo,2025-04-06T11:59:20,Chief Secretary to the Treasury Darren Jones h...,2025-04-06 12:00:11,"<html><body><div><div class=""sc-18fde0d6-0 dlW...",2025-04-06 14:00:15,COMPLETE,False
1,78,2,Man killed in shooting at Stanley house,https://www.bbc.com/news/articles/cwy7en200gyo,2025-04-06T10:14:47,Man in 50s killed in shooting at house\n\nSimo...,2025-04-06 12:00:11,"<html><body><div><article><p class=""sc-18fde0d...",2025-04-06 14:00:17,COMPLETE,False
2,80,2,Japanese Grand Prix: Max Verstappen steals Suz...,https://www.bbc.com/sport/formula1/articles/c4...,2025-04-06T11:19:02,McLaren's one chink of light in the race was t...,2025-04-06 12:00:11,"<html><body><div><div class=""ssrcss-7uxr49-Ric...",2025-04-06 14:00:17,COMPLETE,False
3,81,2,Father and daughter who died in Ingoldmells ca...,https://www.bbc.com/news/articles/cqx4932x9vwo,2025-04-06T13:07:56,Two people who died following a fire at a cara...,2025-04-06 13:25:33,"<html><body><div><div class=""sc-18fde0d6-0 dlW...",2025-04-06 15:25:38,COMPLETE,False
4,82,2,UK Weather: Will the sunshine last until Easte...,https://www.bbc.com/weather/articles/c62ge0mv876o,2025-04-06T12:38:09,A shift in the centre of the high pressure sou...,2025-04-06 13:25:33,"<html><body><div><div class=""ssrcss-7uxr49-Ric...",2025-04-06 15:25:43,COMPLETE,False
...,...,...,...,...,...,...,...,...,...,...,...
104,197,3,Judge Says One DOGE Member Can Access Sensitiv...,https://www.nytimes.com/2025/04/11/nyregion/do...,2025-04-12T01:17:08,A Manhattan federal judge ruled on Friday that...,2025-04-12 08:03:30,"<html><body><div><div class=""css-s99gbd StoryB...",2025-04-12 10:22:57,COMPLETE,False
105,198,3,What to Know About U.S. Talks With Iran Over I...,https://www.nytimes.com/2025/04/12/world/middl...,2025-04-12T04:01:16,A brief handshake may be the most likely outco...,2025-04-12 08:03:30,"<html><body><div><div class=""css-s99gbd StoryB...",2025-04-12 10:23:01,COMPLETE,False
106,199,3,Has Disney+ Changed ‘Doctor Who’? U.S. and U.K...,https://www.nytimes.com/2025/04/11/arts/televi...,2025-04-11T04:01:09,"Last year, when Disney+ was spending big to pr...",2025-04-12 08:03:30,"<html><body><div><div class=""css-s99gbd StoryB...",2025-04-12 10:23:01,COMPLETE,False
107,200,3,Hudson River Helicopter Crash Is a Tragic End ...,https://www.nytimes.com/2025/04/11/nyregion/se...,2025-04-11T21:13:28,“Living the dream.”\n\nThat was how Seankese J...,2025-04-12 08:03:30,"<html><body><div><div class=""css-s99gbd StoryB...",2025-04-12 10:23:01,COMPLETE,False


In [28]:
query

"Aruba|Afghanistan|Angola|Anguilla|Åland Islands|Albania|Andorra|United Arab Emirates|Argentina|Armenia|American Samoa|Antarctica|French Southern Territories|Antigua and Barbuda|Australia|Austria|Azerbaijan|Burundi|Belgium|Benin|Bonaire, Sint Eustatius and Saba|Burkina Faso|Bangladesh|Bulgaria|Bahrain|Bahamas|Bosnia and Herzegovina|Saint Barthélemy|Belarus|Belize|Bermuda|Bolivia, Plurinational State of|Brazil|Barbados|Brunei Darussalam|Bhutan|Bouvet Island|Botswana|Central African Republic|Canada|Cocos (Keeling) Islands|Switzerland|Chile|China|Côte d'Ivoire|Cameroon|Congo, The Democratic Republic of the|Congo|Cook Islands|Colombia|Comoros|Cabo Verde|Costa Rica|Cuba|Curaçao|Christmas Island|Cayman Islands|Cyprus|Czechia|Germany|Djibouti|Dominica|Denmark|Dominican Republic|Algeria|Ecuador|Egypt|Eritrea|Western Sahara|Spain|Estonia|Ethiopia|Finland|Fiji|Falkland Islands (Malvinas)|France|Faroe Islands|Micronesia, Federated States of|Gabon|United Kingdom|Georgia|Guernsey|Ghana|Gibraltar|Gu

In [32]:
len(df['content'][34])

874

In [18]:
def filter_content():
    for index, row in df.iterrows():
        if row['status']=="COMPLETED":
        


,id,source_id,title,url,published_date,content,created_at,html_content,extraction_date,status
0,1,1,Dominion still has pending lawsuits against el...,https://www.cnn.com/business/live-news/fox-new...,None,None,2025-04-06 09:15:11,None,None,FAILED
1,2,1,Russia is 'going backwards' in equipment and d...,https://www.cnn.com/europe/live-news/russia-uk...,None,None,2025-04-06 09:15:11,None,None,FAILED
2,3,1,Podcast: One country musician is calling for o...,https://www.cnn.com/audio/podcasts/the-assignm...,None,None,2025-04-06 09:15:11,None,None,FAILED
3,4,1,Bidets save you money and reduce waste — we te...,https://www.cnn.com/cnn-underscored/reviews/be...,None,None,2025-04-06 09:15:11,None,None,FAILED
4,5,1,50+ products to make your life easier and our ...,https://www.cnn.com/cnn-underscored/home/edito...,None,None,2025-04-06 09:15:11,None,None,FAILED
...,...,...,...,...,...,...,...,...,...,...
97,98,3,The Future of Baseball - The New York Times,https://www.nytimes.com/2025/04/06/briefing/ba...,2025-04-06T11:16:53,"A new baseball season is underway, and the spo...",2025-04-06 13:25:36,"<html><body><div><div class=""css-s99gbd StoryB...",2025-04-06 15:25:52,COMPLETE
98,99,3,Recovering Pope Francis Surprises Pilgrims Wit...,https://www.nytimes.com/2025/04/06/world/europ...,2025-04-06T12:32:47,"As entrances go, this one was both unexpected ...",2025-04-06 13:25:36,"<html><body><div><div class=""css-s99gbd StoryB...",2025-04-06 15:25:52,COMPLETE
99,100,3,Long-Running Storm Drenches Central U.S. but S...,https://www.nytimes.com/2025/04/06/weather/sto...,2025-04-06T12:27:55,The huge storm system that has caused widespre...,2025-04-06 13:25:36,"<html><body><div><div class=""css-s99gbd StoryB...",2025-04-06 15:25:52,COMPLETE
100,101,3,This Agency Fights Corruption. New York City L...,https://www.nytimes.com/2025/04/06/nyregion/ny...,2025-04-06T07:00:06,"In recent months, New York City’s government h...",2025-04-06 13:25:36,"<html><body><div><div class=""css-s99gbd StoryB...",2025-04-06 15:25:52,COMPLETE


In [13]:
import transformers
from transformers import pipeline
import accelerate
print(f"Transformers version: {accelerate.__version__}")

# Try reinstalling transformers with specific version
#!pip install transformers==4.30.2 --force-reinstall


Transformers version: 1.6.0


In [15]:
import torch

In [18]:
!uv add ollama

Resolved 88 packages in 573ms                                        
⠙ Preparing packages... (0/10)                                                  ⠋ Preparing packages... (0/0)                                                   
⠙ Preparing packages... (0/10)---     0 B/56.89 KiB                     
annotated-types ------------------------------     0 B/13.32 KiB
⠙ Preparing packages... (0/10)---     0 B/56.89 KiB                     
annotated-types ------------------------------     0 B/13.32 KiB
⠙ Preparing packages... (0/10)--- 14.84 KiB/56.89 KiB                   
annotated-types ------------------------------ 13.32 KiB/13.32 KiB
⠙ Preparing packages... (0/10)--- 14.84 KiB/56.89 KiB                   
⠙ Preparing packages... (0/10)--- 14.84 KiB/56.89 KiB                   
h11        ------------------------------ 14.84 KiB/56.89 KiB
⠙ Preparing packages... (0/10)---     0 B/432.91 KiB                    
h11        ------------------------------ 14.84 KiB/56.89 KiB
anyio     

In [16]:
print(torch.cuda.is_available()) 

True


In [17]:
from transformers import pipeline
classifier = pipeline("zero-shot-classification",
                      model="facebook/bart-large-mnli")

NameError: name 'init_empty_weights' is not defined